# Getting Started with Quantumium

This notebook introduces the basics of quantum computing with the Quantumium SDK.
You'll learn to create quantum circuits, apply gates, simulate statevectors,
and measure outcomes.

## Installation

```bash
pip install quantumium
```

## 1. Your First Qubit

A qubit starts in the |0⟩ state. Let's create a single-qubit circuit and inspect its statevector.

In [1]:
from quantumium.core.circuit import QuantumCircuit
from quantumium.core.gate import H, X, Y, Z, CNOT

# Single qubit, initialized to |0>
qc = QuantumCircuit(1)
sv = qc.statevector()
print(f"Initial state: {sv}")
print(f"This is |0> = [1, 0]")

Initial state: [1.+0.j 0.+0.j]
This is |0> = [1, 0]


## 2. Quantum Gates

Gates transform qubit states. The **X gate** flips |0⟩ to |1⟩ (quantum NOT).
The **Hadamard (H) gate** creates superposition.

In [2]:
# X gate: |0> -> |1>
qc_x = QuantumCircuit(1)
qc_x.apply(X, 0)
print(f"After X gate: {qc_x.statevector()}")

# H gate: |0> -> |+> = (|0> + |1>)/sqrt(2)
qc_h = QuantumCircuit(1)
qc_h.apply(H, 0)
sv = qc_h.statevector()
print(f"After H gate: {sv}")
print(f"Equal superposition: |0> and |1> each have amplitude {sv[0]:.4f}")

After X gate: [0.+0.j 1.+0.j]
After H gate: [0.70710678+0.j 0.70710678+0.j]
Equal superposition: |0> and |1> each have amplitude 0.7071+0.0000j


## 3. Measurement

When we measure a qubit in superposition, we get random outcomes
with probabilities determined by the amplitudes.

In [3]:
qc = QuantumCircuit(1)
qc.apply(H, 0)

counts = qc.measure_all(shots=1000)
print(f"Measurement results (1000 shots): {counts}")
print(f"Roughly 50/50 between |0> and |1>")

Measurement results (1000 shots): {'0': 503, '1': 497}
Roughly 50/50 between |0> and |1>


## 4. Multi-Qubit Circuits: The Bell State

The Bell state is the simplest entangled state. Two qubits are correlated:
measuring one instantly determines the other.

In [4]:
qc = QuantumCircuit(2)
qc.apply(H, 0)        # Superposition on qubit 0
qc.apply(CNOT, 0, 1)  # Entangle qubits 0 and 1

sv = qc.statevector()
print(f"Bell state: {sv}")
print(f"Only |00> and |11> have non-zero amplitude")

counts = qc.measure_all(shots=1000)
print(f"\nMeasurement: {counts}")
print("Always correlated: both 0 or both 1")

Bell state: [0.70710678+0.j 0.        +0.j 0.        +0.j 0.70710678+0.j]
Only |00> and |11> have non-zero amplitude

Measurement: {'00': 510, '11': 490}
Always correlated: both 0 or both 1


## 5. Circuit Visualization

Quantumium provides ASCII circuit diagrams.

In [5]:
from quantumium.utils.visualization import circuit_to_ascii

qc = QuantumCircuit(3)
qc.apply(H, 0)
qc.apply(CNOT, 0, 1)
qc.apply(CNOT, 0, 2)

print("GHZ state circuit:")
print(circuit_to_ascii(qc))

GHZ state circuit:
q0: ──H──●──●──
         │  │
q1: ─────⊕──│──
            │
q2: ────────⊕──


## 6. Parameterized Circuits

For variational algorithms (VQE, QAOA), you need circuits with tunable parameters.

In [6]:
from quantumium.core.parameter import Parameter, ParameterizedCircuit
import numpy as np

theta = Parameter("theta")
pqc = ParameterizedCircuit(1)
pqc.ry(theta, 0)

for angle in [0, np.pi/4, np.pi/2, np.pi]:
    bound = pqc.bind({theta: angle})
    sv = bound.statevector()
    prob_1 = abs(sv[1])**2
    print(f"theta={angle:.4f}: P(|1>) = {prob_1:.4f}")

theta=0.0000: P(|1>) = 0.0000
theta=0.7854: P(|1>) = 0.1464
theta=1.5708: P(|1>) = 0.5000
theta=3.1416: P(|1>) = 1.0000


## 7. OpenQASM Export

Export circuits to OpenQASM 2.0 for interoperability with other frameworks.

In [7]:
from quantumium.transpiler.qasm import to_qasm

qc = QuantumCircuit(2)
qc.apply(H, 0)
qc.apply(CNOT, 0, 1)

qasm = to_qasm(qc)
print(qasm)

OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
creg c[2];

h q[0];
cx q[0],q[1];



## Next Steps

- **02_quantum_algorithms.ipynb** — Grover's search, Deutsch-Jozsa
- **03_vqe_chemistry.ipynb** — Molecular simulation with VQE
- **04_qplus_factoring.ipynb** — Factor integers with Q+
- **05_noise_and_error_mitigation.ipynb** — Realistic noisy simulation